# Outlines 101 — Guaranteed Structured Generation

**Week 2 | Notebook 1 of 4**

**What you'll learn:**
- Why LLMs fail at structured output
- FSA-based token-level constraint enforcement
- Pydantic model output — always valid JSON
- JSON Schema, Regex, and Choice generation
- Adherence rate benchmark vs raw prompting

**Runtime:** ~35 minutes

**Hardware:** GPU recommended for local models, but CPU works for small models

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("02_outlines/01_token_constraints.ipynb")

## 1. The Problem: Why LLMs Fail at Structured Output

In [ ]:
from src.config import get_model, get_openai_client

client = get_openai_client()

# Try raw prompting for structured JSON — often fails
response = client.chat.completions.create(
    model=get_model(),
    messages=[
        {"role": "system", "content": 'Return ONLY JSON: {"name": string, "age": int}'},
        {"role": "user", "content": "Extract: Jane, 28"},
    ],
)

print("Raw output:")
print(response.choices[0].message.content)



## 2. Setup — Two Backends

- **JSON-schema constraints** → hosted provider backend (structured outputs: OpenAI, Gemini, or Groq)
- **Regex / Choice** (token-level masking) → local transformers model (a hosted API can't mask logits)


In [ ]:
from typing import Literal

import outlines
from pydantic import BaseModel, Field

from src.config import get_outlines_model

# JSON-schema constraints go through the selected provider's OpenAI-compatible
# endpoint (LLM_PROVIDER in .env). Regex/Choice/CFG need token-level masking —
# those cells load a local model instead.
# Note: hard guarantees require the endpoint to honor structured outputs
# (OpenAI and Groq's gpt-oss do; Llama on Groq degrades to prompt-only constraints).
model = get_outlines_model()


## 3. Pydantic Model Output — Customer Support Ticket

In [ ]:
class SupportTicket(BaseModel):
    customer_name: str
    priority: Literal["low", "medium", "high", "critical"]
    category: Literal["billing", "technical", "shipping", "other"]
    summary: str = Field(max_length=200)
    escalate_to_human: bool


ticket_text = """
Customer: Sarah Miller
Issue: My payment failed 3 times and my account is now locked.
I need access restored urgently as I have a deadline tomorrow.
"""

# Outlines generates JSON that ALWAYS validates against the Pydantic model
result = model(f"Extract a support ticket from this message: {ticket_text}", SupportTicket)

# result is a JSON string — cast it
ticket = SupportTicket.model_validate_json(result)
print(ticket.model_dump_json(indent=2))

## 4. JSON Schema Output — Complex Nested Schema

In [ ]:
import json

from outlines.types import JsonSchema

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer", "minimum": 0, "maximum": 150},
        "email": {"type": "string", "format": "email"},
        "tags": {"type": "array", "items": {"type": "string"}, "maxItems": 5},
        "status": {"enum": ["active", "inactive", "pending"]},
    },
    # OpenAI structured outputs require `required` to list EVERY property
    # (no optional keys) — optional-looking fields use nullable types instead.
    "required": ["name", "age", "email", "tags", "status"],
    "additionalProperties": False,
}

result = model(
    "Create a user profile for a software engineer named Alex, age 30, active status",
    JsonSchema(json.dumps(schema)),
)

print(result)


## 5. Regex Output — Phone Numbers, Dates, IDs

In [ ]:
import outlines
import torch
from outlines.types import Regex
from transformers import AutoModelForCausalLM, AutoTokenizer

# Token-level masking (regex, choice, CFG) needs control over the logits, which a
# hosted API does not expose — so we run a small local model here.
# First run downloads ~1 GB (Qwen2.5-0.5B); it's cached afterwards.
LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
hf_model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=torch.float32)
local_model = outlines.from_transformers(hf_model, tokenizer)

# Phone number
phone = local_model(
    "Extract phone number from: 'Contact us at +91-98765-43210'", Regex(r"\+\d{1,3}-\d{5}-\d{5}")
)
print(f"Phone: {phone}")

# Date
date = local_model("Invoice date:", Regex(r"\d{4}-\d{2}-\d{2}"))
print(f"Date: {date}")

# UUID
uuid = local_model("ID:", Regex(r"[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}"))
print(f"UUID: {uuid}")




## 6. Choice Output — Zero-Hallucination Classification

In [ ]:
from outlines.types import Choice

# The model CANNOT output anything outside this list.
# Uses local_model from the previous cell — Choice requires token-level masking,
# which hosted APIs (OpenAI etc.) do not expose.
category = local_model(
    "Classify this support ticket: 'My payment failed'",
    Choice(["billing", "technical", "shipping", "account", "other"]),
)
print(f"Category: {category}")


## 7. Adherence Rate Benchmark: Outlines vs Raw Prompting

In [ ]:
from src.datasets import generate_invoice_texts

invoices = generate_invoice_texts(10)


class SimpleInvoice(BaseModel):
    invoice_number: str
    customer_name: str
    total_amount: float
    payment_status: Literal["paid", "pending", "overdue"]


outlines_valid = 0
raw_valid = 0

for inv in invoices:
    # Outlines method
    try:
        result = model(f"Extract invoice: {inv['raw_text']}", SimpleInvoice)
        parsed = SimpleInvoice.model_validate_json(result)
        outlines_valid += 1
    except Exception:
        pass

    # Raw prompting method
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "Return ONLY JSON with keys: invoice_number, customer_name, total_amount, payment_status",
                },
                {"role": "user", "content": f"Extract: {inv['raw_text']}"},
            ],
        )
        text = response.choices[0].message.content
        # Try to extract JSON from possible markdown
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        parsed_raw = SimpleInvoice.model_validate_json(text.strip())
        raw_valid += 1
    except Exception:
        pass

print(f"Outlines adherence: {outlines_valid}/10 ({outlines_valid * 10}%)")
print(f"Raw prompting adherence: {raw_valid}/10 ({raw_valid * 10}%)")